# Lab 02. 기술통계, 표준편차와 z점수

평균은 중심을, 분산과 표준편차는 퍼짐을 나타낸다. z점수는 관측값이 평균에서 표준편차 몇 배 떨어져 있는지 표현한다.
z점수가 크다는 사실만으로 좋은 지역 또는 오류라고 판단할 수 없다.

In [ ]:
from pathlib import Path
import sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "src").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 루트에서 Notebook을 실행하세요.")

sys.path.insert(0, str(ROOT))
STUDENT_ID = "20260001"  # 반드시 본인 학번으로 변경
print("저장소:", ROOT)
print("실습 학번:", STUDENT_ID)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.education.personalized_data import make_student_dataset

df = make_student_dataset(STUDENT_ID).drop_duplicates()
df["유동인구"] = df.groupby("업종")["유동인구"].transform(lambda s: s.fillna(s.median()))
df["월임대료"] = df.groupby("업종")["월임대료"].transform(lambda s: s.fillna(s.median()))

## 1. 작은 데이터로 공식을 검증

먼저 다섯 값으로 편차, 편차제곱과 표본분산을 한 단계씩 확인한다.

In [ ]:
x = np.array([2, 4, 4, 5, 10], dtype=float)
x_bar = x.sum() / len(x)
table = pd.DataFrame({"x": x, "편차": x - x_bar, "편차제곱": (x - x_bar) ** 2})
sample_variance = table["편차제곱"].sum() / (len(x) - 1)
sample_std = np.sqrt(sample_variance)

display(table)
print("평균:", x_bar, "표본분산:", sample_variance, "표본표준편차:", sample_std)
assert np.isclose(sample_variance, np.var(x, ddof=1))

## 2. 학생별 데이터 기술통계

평균과 중앙값의 차이가 크면 비대칭 분포나 이상값을 의심할 수 있다.

In [ ]:
numeric = ["상주인구", "유동인구", "경쟁점포수", "주차장수", "월임대료", "월매출"]
summary = df[numeric].agg(["count", "mean", "median", "std", "min", "max"]).T
summary["평균-중앙값"] = summary["mean"] - summary["median"]
display(summary.round(2))

## 3. z점수 직접 계산

pandas 표본표준편차 ddof=1을 사용한다. 절댓값 3 초과는 확인 후보일 뿐 자동 삭제 규칙이 아니다.

In [ ]:
mean_sales = df["월매출"].mean()
std_sales = df["월매출"].std(ddof=1)
df["매출_z"] = (df["월매출"] - mean_sales) / std_sales
extreme = df.loc[df["매출_z"].abs() > 3].sort_values("매출_z", key=abs, ascending=False)
print("z점수 평균:", df["매출_z"].mean())
print("z점수 표준편차:", df["매출_z"].std(ddof=1))
display(extreme)

## 4. 업종별 기술통계

전체 평균은 업종 구성비의 영향을 받는다. 집단별 개수도 함께 봐야 한다.

In [ ]:
group_summary = (
    df.groupby("업종")["월매출"]
      .agg(개수="count", 평균="mean", 중앙값="median", 표준편차="std")
      .sort_values("평균", ascending=False)
)
group_summary["변동계수"] = group_summary["표준편차"] / group_summary["평균"]
display(group_summary.round(2))

## 5. 독립 연습

1. 월임대료의 평균, 표본분산, 표본표준편차를 NumPy의 mean, var, std 없이 계산한다.
2. 유동인구 z점수와 경쟁점포수 z점수를 계산한다.
3. 업종 안에서 계산한 z점수와 전체에서 계산한 z점수가 가장 크게 다른 행 5개를 찾는다.
4. 어떤 기준의 z점수가 입지 판단에 더 적절한지 설명한다.

In [ ]:
# TODO 1
rent = df["월임대료"].to_numpy()
manual_mean = None
manual_variance = None
manual_std = None

# TODO 2~3
df["유동인구_z_전체"] = None
df["유동인구_z_업종"] = None
largest_difference = None
display(largest_difference)

## 6. 자가점검

- [ ] 분산의 단위와 표준편차의 단위를 구분한다.
- [ ] ddof=0과 ddof=1을 코드로 비교했다.
- [ ] 평균·중앙값 차이를 분포와 연결했다.
- [ ] 전체 z점수와 집단 내 z점수의 질문이 다름을 설명한다.